In [9]:
import folium
import pandas as pd
import json

# ══════════════════════════════════════════════════════════════
# 1. Carica i quattro file
# ══════════════════════════════════════════════════════════════

df_mh    = pd.read_csv('MeteoHub_Adriatic/adriatic_coast_stations_aggregated.csv')
df_fvg   = pd.read_csv('DPC_friuli/DPC_latest.csv')
df_arpae = pd.read_csv('ARPAE_emilia/ARPAE_coastal_latest.csv')
df_arpav = pd.read_csv('ARPAV_veneto/ARPAV_latest.csv')

# ══════════════════════════════════════════════════════════════
# 2. Normalizza ciascuna fonte a (station_id, name, lat, lon, source)
#    una riga per stazione, niente duplicati
# ══════════════════════════════════════════════════════════════

# --- MeteoHub: già wide, una riga per stazione ---
mh_stations = df_mh[['station_id', 'sensor_name', 'lat', 'lon']].drop_duplicates('station_id').copy()
mh_stations['source'] = 'MeteoHub (Veneto/Marche/Puglia)'
mh_stations = mh_stations.rename(columns={'sensor_name': 'name'})

# --- FVG: long format, dedup su station_id ---
fvg_stations = df_fvg[['station_id', 'station_name', 'lat', 'lon']].drop_duplicates('station_id').copy()
fvg_stations['source'] = 'DPC FVG'
fvg_stations = fvg_stations.rename(columns={'station_name': 'name'})

# --- ARPAE: long format, dedup su station_id ---
arpae_stations = df_arpae[['station_id', 'sensor_name', 'lat', 'lon']].drop_duplicates('station_id').copy()
arpae_stations['source'] = 'ARPAE Emilia-Romagna'
arpae_stations = arpae_stations.rename(columns={'sensor_name': 'name'})

# --- ARPAV: long format, dedup su station_id ---
arpav_stations = df_arpav[['station_id', 'sensor_name', 'lat', 'lon']].drop_duplicates('station_id').copy()
arpav_stations['source'] = 'ARPAV Veneto'
arpav_stations = arpav_stations.rename(columns={'sensor_name': 'name'})

# ══════════════════════════════════════════════════════════════
# 3. Unisci tutto
# ══════════════════════════════════════════════════════════════

all_stations = pd.concat(
    [mh_stations, fvg_stations, arpae_stations, arpav_stations],
    ignore_index=True
)
all_stations = all_stations.dropna(subset=['lat', 'lon'])

print(f'Totale stazioni sulla mappa: {len(all_stations)}')
print(all_stations['source'].value_counts().to_string())

# ══════════════════════════════════════════════════════════════
# 4. Mappa Folium
# ══════════════════════════════════════════════════════════════

SOURCE_COLORS = {
    'MeteoHub (Veneto/Marche/Puglia)': 'blue',
    'DPC FVG': 'red',
    'ARPAE Emilia-Romagna': 'green',
    'ARPAV Veneto': 'purple',
}

map_center = [all_stations['lat'].mean(), all_stations['lon'].mean()]
m = folium.Map(location=map_center, zoom_start=7, tiles='CartoDB positron')

for _, row in all_stations.iterrows():
    color = SOURCE_COLORS.get(row['source'], 'gray')
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=f"<b>{row['name']}</b><br>Fonte: {row['source']}",
    ).add_to(m)

# Legenda
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border-radius: 6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.3); font-size: 13px;">
  <b>Fonte dati</b><br>
  <span style="color:blue;">●</span> MeteoHub (Veneto/Marche/Puglia)<br>
  <span style="color:red;">●</span> DPC FVG<br>
  <span style="color:green;">●</span> ARPAE Emilia-Romagna<br>
  <span style="color:purple;">●</span> ARPAV Veneto
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

display(m)

Totale stazioni sulla mappa: 865
source
DPC FVG                            340
ARPAV Veneto                       245
ARPAE Emilia-Romagna               158
MeteoHub (Veneto/Marche/Puglia)    122
